# Echo Epoch Metric Aggregation

Aggregate the per-video SAMWISE Echo epoch scoring outputs into summaries by class, view, and dataset.

This notebook expects the outputs produced by `scripts/eval_echo_epoch_p2flow_metrics.py`.

In [2]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)


In [4]:
EXP_DIR = Path('output/echo_refvos_train')
EPOCH = 1

EPOCH_DIR = EXP_DIR / f'valid_epoch{EPOCH:02d}'
VALID_DIR = EPOCH_DIR / 'eval_echo' / 'valid'
CSV_PATH = VALID_DIR / 'metrics_p2flow_per_video.csv'
JSON_PATH = VALID_DIR / 'metrics_p2flow_per_video.json'

print('EPOCH_DIR:', EPOCH_DIR.resolve())
print('CSV_PATH exists:', CSV_PATH.exists())
print('JSON_PATH exists:', JSON_PATH.exists())

EPOCH_DIR: /home/ultrai/UltrAi/moein/SAMWISE/output/echo_refvos_train/valid_epoch01
CSV_PATH exists: True
JSON_PATH exists: True


In [5]:
def load_metric_rows(csv_path: Path, json_path: Path) -> pd.DataFrame:
    if csv_path.exists():
        df = pd.read_csv(csv_path)
    elif json_path.exists():
        payload = json.loads(json_path.read_text())
        rows = []
        for video_id, video_payload in payload.get('videos', {}).items():
            rows.extend(video_payload.get('records', []))
        df = pd.DataFrame(rows)
    else:
        raise FileNotFoundError(f'Neither CSV nor JSON exists: {csv_path} | {json_path}')

    if df.empty:
        raise ValueError('Loaded metric table is empty.')

    for col in ['video_id', 'dataset', 'view', 'clip_id', 'exp_id', 'obj_id', 'class_name', 'frame']:
        if col in df.columns:
            df[col] = df[col].astype(str)

    for col in ['dice', 'iou', 'hd95']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    return df


df = load_metric_rows(CSV_PATH, JSON_PATH)
print('Rows:', len(df))
print('Videos:', df['video_id'].nunique())
print('Datasets:', sorted(df['dataset'].unique().tolist()))
print('Views:', sorted(df['view'].unique().tolist()))
print('Classes:', sorted(df['class_name'].unique().tolist()))
display(df.head())

Rows: 8698
Videos: 969
Datasets: ['Camus', 'CardiacUDA', 'EchoCP', 'EchoNet-Pediatric', 'SegRWMA']
Views: ['2CH', '3CH', '4CH', 'PSAX']
Classes: ['LA', 'LV', 'MYO', 'RA', 'RV']


,video_id,dataset,view,clip_id,exp_id,obj_id,class_name,frame,dice,iou,hd95
0,Camus__2CH__patient0023,Camus,2CH,patient0023,0,1,LV,0,0.909263,0.833623,9.000000
1,Camus__2CH__patient0023,Camus,2CH,patient0023,0,1,LV,1,0.918715,0.849651,8.000000
2,Camus__2CH__patient0023,Camus,2CH,patient0023,0,1,LV,2,0.918768,0.849742,8.544004
3,Camus__2CH__patient0023,Camus,2CH,patient0023,0,1,LV,3,0.898683,0.816007,9.848858
4,Camus__2CH__patient0023,Camus,2CH,patient0023,0,1,LV,4,0.900800,0.819505,9.848858


In [6]:
def summarize_group(frame_df: pd.DataFrame) -> pd.Series:
    out = {'n_rows': int(len(frame_df))}
    for metric in ['dice', 'iou', 'hd95']:
        values = frame_df[metric].replace([np.inf, -np.inf], np.nan).dropna()
        out[f'{metric}_n'] = int(values.shape[0])
        out[f'{metric}_mean'] = float(values.mean()) if len(values) else np.nan
        out[f'{metric}_std'] = float(values.std(ddof=0)) if len(values) else np.nan
    return pd.Series(out)


def summarize_by(df: pd.DataFrame, group_cols: list[str]) -> pd.DataFrame:
    summary = df.groupby(group_cols, dropna=False, sort=True).apply(summarize_group).reset_index()
    return summary.sort_values(group_cols).reset_index(drop=True)


def add_macro_means(summary_df: pd.DataFrame, metric_prefixes=('dice', 'iou', 'hd95')) -> pd.DataFrame:
    df_out = summary_df.copy()
    for metric in metric_prefixes:
        mean_col = f'{metric}_mean'
        if mean_col in df_out.columns:
            values = pd.to_numeric(df_out[mean_col], errors='coerce')
            df_out.attrs[f'mean_{metric}'] = float(values.dropna().mean()) if values.notna().any() else np.nan
    return df_out


## Per Class

In [7]:
per_class = add_macro_means(summarize_by(df, ['class_name']))
display(per_class)
print({k: v for k, v in per_class.attrs.items() if k.startswith('mean_')})

,class_name,n_rows,dice_n,dice_mean,dice_std,iou_n,iou_mean,iou_std,hd95_n,hd95_mean,hd95_std
0,LA,2212.0,2212.0,0.861121,0.178580,2212.0,0.784585,0.184976,2212.0,12.194466,20.510282
1,LV,4065.0,4065.0,0.855686,0.200504,4065.0,0.782000,0.198116,3987.0,10.678667,21.207357
2,MYO,2000.0,2000.0,0.849630,0.054358,2000.0,0.742263,0.078318,2000.0,7.472176,3.605892
3,RA,208.0,208.0,0.806775,0.249696,208.0,0.725650,0.239815,208.0,28.425874,34.298210
4,RV,213.0,213.0,0.763536,0.246353,213.0,0.663028,0.231858,213.0,38.523559,41.696126


{'mean_dice': 0.8273495779938511, 'mean_iou': 0.7395050251750414, 'mean_hd95': 19.458948103590565}


## Per View

In [8]:
per_view = summarize_by(df, ['view'])
display(per_view)

,view,n_rows,dice_n,dice_mean,dice_std,iou_n,iou_mean,iou_std,hd95_n,hd95_mean,hd95_std
0,2CH,3096.0,3096.0,0.869781,0.125673,3096.0,0.784663,0.139424,3076.0,8.988576,10.781574
1,3CH,96.0,96.0,0.539726,0.371999,96.0,0.452932,0.329403,75.0,38.037049,32.133640
2,4CH,4615.0,4615.0,0.839278,0.203004,4615.0,0.757606,0.200868,4578.0,14.047965,25.713421
3,PSAX,891.0,891.0,0.892189,0.060090,891.0,0.810177,0.089188,891.0,4.264009,2.496704


## Per Dataset

In [9]:
per_dataset = summarize_by(df, ['dataset'])
display(per_dataset)

,dataset,n_rows,dice_n,dice_mean,dice_std,iou_n,iou_mean,iou_std,hd95_n,hd95_mean,hd95_std
0,Camus,6000.0,6000.0,0.888049,0.071968,6000.0,0.805016,0.100095,6000.0,7.358195,4.766005
1,CardiacUDA,622.0,622.0,0.735436,0.301823,622.0,0.648222,0.282397,616.0,39.664284,45.084174
2,EchoCP,223.0,223.0,0.453883,0.413795,223.0,0.391250,0.366438,210.0,55.275961,45.648228
3,EchoNet-Pediatric,1553.0,1553.0,0.886015,0.068297,1553.0,0.801286,0.097448,1553.0,4.701217,2.986531
4,SegRWMA,300.0,300.0,0.499775,0.384435,300.0,0.420274,0.342372,241.0,46.145825,41.804679


## Per View and Class

In [10]:
per_view_class = summarize_by(df, ['view', 'class_name'])
display(per_view_class)

,view,class_name,n_rows,dice_n,dice_mean,dice_std,iou_n,iou_mean,iou_std,hd95_n,hd95_mean,hd95_std
0,2CH,LA,1000.0,1000.0,0.885927,0.110114,1000.0,0.807933,0.131279,1000.0,8.088848,5.948382
1,2CH,LV,1096.0,1096.0,0.876510,0.171513,1096.0,0.806009,0.171742,1076.0,10.187356,16.741236
2,2CH,MYO,1000.0,1000.0,0.846260,0.060559,1000.0,0.737999,0.086044,1000.0,8.598416,4.254492
3,3CH,LV,96.0,96.0,0.539726,0.371999,96.0,0.452932,0.329403,75.0,38.037049,32.133640
4,4CH,LA,1212.0,1212.0,0.840654,0.217423,1212.0,0.765321,0.217730,1212.0,15.581939,26.705500
5,4CH,LV,1982.0,1982.0,0.843064,0.228093,1982.0,0.771995,0.222120,1945.0,12.834055,25.917611
6,4CH,MYO,1000.0,1000.0,0.853000,0.047111,1000.0,0.746527,0.069479,1000.0,6.345936,2.316761
7,4CH,RA,208.0,208.0,0.806775,0.249696,208.0,0.725650,0.239815,208.0,28.425874,34.298210
8,4CH,RV,213.0,213.0,0.763536,0.246353,213.0,0.663028,0.231858,213.0,38.523559,41.696126
9,PSAX,LV,891.0,891.0,0.892189,0.060090,891.0,0.810177,0.089188,891.0,4.264009,2.496704


## Per Dataset and Class

In [11]:
per_dataset_class = summarize_by(df, ['dataset', 'class_name'])
display(per_dataset_class)

,dataset,class_name,n_rows,dice_n,dice_mean,dice_std,iou_n,iou_mean,iou_std,hd95_n,hd95_mean,hd95_std
0,Camus,LA,2000.0,2000.0,0.892801,0.093818,2000.0,0.816246,0.118869,2000.0,7.986412,6.650772
1,Camus,LV,2000.0,2000.0,0.921715,0.033902,2000.0,0.856538,0.055454,2000.0,6.615997,3.154468
2,Camus,MYO,2000.0,2000.0,0.849630,0.054358,2000.0,0.742263,0.078318,2000.0,7.472176,3.605892
3,CardiacUDA,LA,156.0,156.0,0.714585,0.308042,156.0,0.624555,0.289696,156.0,42.722154,49.798868
4,CardiacUDA,LV,156.0,156.0,0.698770,0.325849,156.0,0.610699,0.296656,150.0,39.266338,44.575914
5,CardiacUDA,RA,152.0,152.0,0.796022,0.286030,152.0,0.725099,0.271621,152.0,31.533048,37.659704
6,CardiacUDA,RV,158.0,158.0,0.733938,0.275868,158.0,0.634681,0.255741,158.0,44.845370,46.101323
7,EchoCP,LA,56.0,56.0,0.137882,0.249530,56.0,0.099616,0.191776,56.0,77.440670,25.353827
8,EchoCP,LV,56.0,56.0,0.000172,0.001196,56.0,0.000086,0.000601,43.0,117.017823,30.379958
9,EchoCP,RA,56.0,56.0,0.835962,0.091367,56.0,0.727145,0.115566,56.0,19.992115,20.554461


## Optional Exports

In [12]:
EXPORT_DIR = VALID_DIR / 'aggregated_tables'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

per_class.to_csv(EXPORT_DIR / 'per_class.csv', index=False)
per_view.to_csv(EXPORT_DIR / 'per_view.csv', index=False)
per_dataset.to_csv(EXPORT_DIR / 'per_dataset.csv', index=False)
per_view_class.to_csv(EXPORT_DIR / 'per_view_class.csv', index=False)
per_dataset_class.to_csv(EXPORT_DIR / 'per_dataset_class.csv', index=False)

print('Saved aggregated tables to', EXPORT_DIR.resolve())

Saved aggregated tables to /home/ultrai/UltrAi/moein/SAMWISE/output/echo_refvos_train/valid_epoch01/eval_echo/valid/aggregated_tables
